Exercise 1: Indexing
Use Elasticsearch to index and query the grocery dataset (directory data/grocery). The dataset contains
a catalogue of items sold in a store. Indications on how to perform the indexing and querying are
provided next. Write your code in a single Python notebook. You are free to use the Elasticsearch
wrapper API, the custom helpers we have used in class, or both. For each of the points below, write
some brief comments (or text cells) in your notebook to describe how you addressed that specific point.
Use the numbering provided below to refer to the points in your notebook (e.g., “Point #1: I took care
of efficient loading by ...”)
Indexing. Please index the dataset taking into account the following points:
1. The dataset needs to be loaded efficiently.
2. This is intended to be a rather dynamic database, where updates are done multiple times per hour
3. Create an index with 2 replicas.
4. The documents should allow full text search on the item name and category name, and should allow
filtering and sorting by category and price.
5. A relevance metric that allows for text length discounting should be used. Increase by 30% the amount
of length discounting compared to the default value.


Querying. For each of the points below, write a separate Python parametric function that performs:


6. An exact-match query on the item name.
7. A full-text query on item name and category, with category boosted by a factor 4 compared to item
name.
8. A full-text query on the category that sorts the results by price (lowest to highest).
9. A fuzzy query on the item name.
Test your functions with at least one example query each.

In [1]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import urllib3 

In [2]:
# Since the docker image uses elasticsearch 8.15.3, my local env also installs a es version 8, 
# as it is not compatible with version 9

# Connect to local Elasticsearch instance
client = Elasticsearch(
  "https://localhost:9200",
  basic_auth=("elastic", "mysecurepassword"),
  verify_certs=False
)

# Should provide a response with a cluster instance and name
client.info()

# Disable warnings caused by not using certificate verification
urllib3.disable_warnings()


c:\Users\mchrn\miniconda3\envs\recsys_exercise_1\Lib\site-packages\elasticsearch\_sync\client\__init__.py:404: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
c:\Users\mchrn\miniconda3\envs\recsys_exercise_1\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
client.indices.delete(index="grocery-index")

ObjectApiResponse({'acknowledged': True})

In [4]:
# To load the dataset efficiently, the data should be loaded in bulk
# Therefore the index should have automatic refreshing disabled
# i.e set refresh_interval: -1 (remember to update it at after data has been inserted) 
client.indices.create(
  index="grocery-index",
  settings={
    "number_of_shards": 4,
    "number_of_replicas": 2,
    "refresh_interval": -1,
    "index": {
      "similarity": {
        "default": {
          "type": "BM25",
          "b": 0.975 # Default value in Elasticsearch is 0,75 https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity
        }
      }
    },
  },
  mappings={
    "properties": {
      "product_name": {
       "type": "text",
       "analyzer": "english"
      },
      "category": {
        "type": "text",
        "analyzer": "english",
        "fields": {
          "raw": {
            "type": "keyword"
          } 
        }
      },
      "price": {
        "type": "float"
      }
    }
  }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'grocery-index'})

In [ ]:
# Verify index has been created with the specified settings and mnappings
client.indices.get(index="grocery-index")

ObjectApiResponse({'grocery-index': {'aliases': {}, 'mappings': {'properties': {'category': {'type': 'text', 'fields': {'raw': {'type': 'keyword'}}, 'analyzer': 'english'}, 'price': {'type': 'float'}, 'product_name': {'type': 'text', 'analyzer': 'english'}}}, 'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'refresh_interval': '-1', 'number_of_shards': '4', 'provided_name': 'grocery-index', 'similarity': {'default': {'type': 'BM25', 'b': '0.975'}}, 'creation_date': '1779724314406', 'number_of_replicas': '2', 'uuid': 'U3Ll72d0Ti2RJHCxsaExSA', 'version': {'created': '8512000'}}}}})

In [6]:
# Create a helper function that reads the csv file and prepares it for a bulk insert into the index
def bulk_index(index_name="grocery-index"):
  df = pd.read_csv("../../data/grocery/products.csv")
  
  # Turn the csv file into a dictionary, so we can fetch the value for each row, by its column key
  # https://www.geeksforgeeks.org/python/pandas-dataframe-to_dict/
  df_dictionary = df.to_dict(orient="records")
  
  # Use python generator function to load the dataset efficiently
  for record in df_dictionary:
    yield {
      "_index": index_name,
      "_id": record['product_id'],
      "_source": {
        "product_name": record['product_name'],
        "aisle_id": record['aisle_id'],
        "department_id": record['department_id'],
        "category": record['category'],
        "price": float(record['price'])
      }
    }

# Code is inspired by: https://stackoverflow.com/questions/71889063/bulk-index-create-documents-with-elasticsearch-for-python 
# & https://www.geeksforgeeks.org/elasticsearch/using-the-elasticsearch-bulk-api-for-high-performance-indexing/
helpers.bulk(client, bulk_index())

(49688, [])

In [ ]:
# Sanity check to see if I actually have all the parameters for an entry in the csv. 
client.get(index="grocery-index", id="10")

ObjectApiResponse({'_index': 'grocery-index', '_id': '10', '_version': 1, '_seq_no': 5, '_primary_term': 1, 'found': True, '_source': {'product_name': 'Sparkling Orange Juice & Prickly Pear Beverage', 'aisle_id': 115, 'department_id': 7, 'category': 'Beverages', 'price': 5.38}})

In [8]:
# Update refresh_interval back to default for ElasticSearch so it now is ready to handle updates done multiple times an hour
client.indices.put_settings(
  index="grocery-index",
  settings={
    "index": {
      "refresh_interval": "5s"
    }
  }
)

ObjectApiResponse({'acknowledged': True})

## Answers to point 1-5
- **Point #1 - The dataset needs to be loaded efficiently**
  - To load the dataset efficiently I loaded the data in bulk. This was done by first disabling the refresh_interval for the index when creating it. Then loading the data into a pandas dataframe that I then turn into a dictionary, which is done so I can easily fetch a value from each row based on the column key, for example category or product_name. I then use a python generator function to build a document per entry in the csv file and use the elasticsearch helper function helpers.bulk(), which handles sending these documents in bulk to Elasticsearch.

- **Point #2 - This is intended to be a rather dynamic database, where updates are done multiple times per hour**
  - After loading the data, I reconfigure the settings for my index and set the refresh_interval to the default 5s, so that the index is refreshed often to reflect updates to it, such that users are not presented with stale data. 

- **Point #3 - Create an index with 2 replicas.**
  - When creating the index, under settings i specify it to make it with 2 replicas in the line: "number_of_replicas": 2,

- **Point #4 - The documents should allow full text search on the item name and category name, and should allow filtering and sorting by category and price.**
  - These are also adresses in the creation of the index, specifically in the "mappings" part. 
  - product_name is set to type: text, which creates an inverted index we can query.
  - price is set to type: keyword, which creates DocValues, which is an "uninverted" index, usefull for sorting, grouping, etc - which is what we want, as we need to use price to filter and sort 
  - category is set to have both type text and a field with type keyword, so we have both an inverted index and DocValues for category.

- **Point #5 - A relevance metric that allows for text length discounting should be used. Increase by 30% the amount of length discounting compared to the default value.**
  - Elasticsearch uses BM25, and its standard values are k1=1.2 and b=0.75.
  - b is the parameter that "Controls to what degree document length normalizes tf values", and so I increase this with 30%: 0.75 + (0.75 * 0.30) = 0.975

## Queries for point 6-9

In [9]:
# Point 6: An exact-match query on the item name
def point_6_exact_match(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    query={
      "match_phrase": {
          "product_name": query_phrase
      }
    }
  )
  print(resp)

point_6_exact_match("grocery-index", "Fresh Breath Oral Rinse Mild Mint")
point_6_exact_match("grocery-index", "Fresh Breath Oral Rinse Mild Mint but not exact")

{'took': 6, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 30.92558, 'hits': [{'_index': 'grocery-index', '_id': '22', '_score': 30.92558, '_source': {'product_name': 'Fresh Breath Oral Rinse Mild Mint', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 8.07}}]}}
{'took': 3, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}}


In [10]:
# Point 7: A full-text query on item name and category, with category boosted by a factor 4 compared to item name
# https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-multi-match-query
def point_7_full_text(index_name, query_phrase, fields=[]):
  resp = client.search( 
    index=index_name,
    query={
      "multi_match": {
        "query": query_phrase,
        "fields": fields
      }
    }
  )
  print(resp)

point_7_full_text("grocery-index", "foods ", ["product_name", "category^4"])
point_7_full_text("grocery-index", "fruits snacks ", ["product_name", "category^4"])

{'took': 3, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 4585, 'relation': 'eq'}, 'max_score': 8.25413, 'hits': [{'_index': 'grocery-index', '_id': '8', '_score': 8.25413, '_source': {'product_name': "Cut Russet Potatoes Steam N' Mash", 'aisle_id': 116, 'department_id': 1, 'category': 'Frozen Foods', 'price': 5.5}}, {'_index': 'grocery-index', '_id': '81', '_score': 8.25413, '_source': {'product_name': 'Blakes Chicken Parmesan Dinner', 'aisle_id': 38, 'department_id': 1, 'category': 'Frozen Foods', 'price': 7.97}}, {'_index': 'grocery-index', '_id': '128', '_score': 8.25413, '_source': {'product_name': 'Organic Wild Blueberries', 'aisle_id': 116, 'department_id': 1, 'category': 'Frozen Foods', 'price': 11.74}}, {'_index': 'grocery-index', '_id': '130', '_score': 8.25413, '_source': {'product_name': 'Vanilla Milk Chocolate Almond Ice Cream Bars Multi-Pack', 'aisle_id': 37, 'department_id': 1, 'category': 'Frozen Foo

In [11]:
# Point 8: A full-text query on the category that sorts the results by price (lowest to highest).
def point_8_full_text(index_name, query_phrase, sorting_parameter):
  resp = client.search(
    index=index_name,
    query={
      "match": {
        "category": query_phrase,
      }
    },
    sort=[
      {
        sorting_parameter: {
          "order": "asc"
        }
      }
    ]
  )
  print(resp)

point_8_full_text("grocery-index", "A cool cold beverage for the summery weather", "price")


{'took': 7, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 6659, 'relation': 'eq'}, 'max_score': None, 'hits': [{'_index': 'grocery-index', '_id': '3689', '_score': None, '_source': {'product_name': 'Beef Broth', 'aisle_id': 69, 'department_id': 15, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '10805', '_score': None, '_source': {'product_name': 'Gold Nutrition Energy Bar Chocolate Peanut Butter', 'aisle_id': 3, 'department_id': 19, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '22265', '_score': None, '_source': {'product_name': 'Lavender with Baking Soda & Alpine Lichen Deodorant Stick', 'aisle_id': 25, 'department_id': 11, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '26370', '_score': None, '_source': {'product_name': 'With Caffeine Peach Mango on the go Drink Mix', 'aisle

In [14]:
# Point 9: A fuzzy query on the item name.
def point_9_fuzzy_query(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    query={
      "match": {
        "product_name": {
          "query": query_phrase,
          "fuzziness": "AUTO"
        }
      }
    }
  )
  print(resp)

point_9_fuzzy_query("grocery-index", "Some rinse for a freasher breath with mint")

{'took': 19, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2990, 'relation': 'eq'}, 'max_score': 14.054338, 'hits': [{'_index': 'grocery-index', '_id': '1726', '_score': 14.054338, '_source': {'product_name': 'Fluoride Rinse - Mint', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 6.33}}, {'_index': 'grocery-index', '_id': '22', '_score': 13.824972, '_source': {'product_name': 'Fresh Breath Oral Rinse Mild Mint', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 8.07}}, {'_index': 'grocery-index', '_id': '37921', '_score': 12.9451275, '_source': {'product_name': 'Original Mild Mint Peroxyl Mouth Sore Rinse', 'aisle_id': 20, 'department_id': 11, 'category': 'Wine & Spirits', 'price': 36.43}}, {'_index': 'grocery-index', '_id': '27624', '_score': 12.303259, '_source': {'product_name': 'Icy Mint Oral Rinse', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal C